In [9]:
# ============================================================
# Cell - Pi Benchmark Results: ONNX Models
# ============================================================

import json
import pandas as pd

with open("pi_results_onnx.json", "r") as f:
    onnx = json.load(f)

print(f"Timestamp : {onnx['timestamp']}")
print(f"Platform  : {onnx['platform']}  |  CPU Cores: {onnx['cpu_cores']}  |  RAM: {onnx['total_ram_mb']:.2f} MB")

models = ["sklearn", "fp32", "int8"]
rows = []
for m in models:
    d = onnx[m]
    lat = d["latency"]
    rows.append({
        "Model"        : m.upper(),
        "Size (KB)"    : d["size_kb"],
        "RAM (KB)"     : d["ram_kb"],
        "Mean (ms)"    : lat["mean_ms"],
        "Std (ms)"     : lat["std_ms"],
        "Min (ms)"     : lat["min_ms"],
        "p50 (ms)"     : lat["p50_ms"],
        "p95 (ms)"     : lat["p95_ms"],
        "p99 (ms)"     : lat["p99_ms"],
        "Max (ms)"     : lat["max_ms"],
    })

onnx_df = pd.DataFrame(rows).set_index("Model")

print("\n── ONNX Model Benchmark ──────────────────────────────────────")
display(onnx_df)

Timestamp : 2026-03-03 07:37:04
Platform  : aarch64  |  CPU Cores: 4  |  RAM: 3796.81 MB

── ONNX Model Benchmark ──────────────────────────────────────


,Size (KB),RAM (KB),Mean (ms),Std (ms),Min (ms),p50 (ms),p95 (ms),p99 (ms),Max (ms)
Model,,,,,,,,,
SKLEARN,90.06,378.0,5.9632,28.8927,1.3985,1.9428,4.2846,233.1406,294.3745
FP32,13.98,288.0,0.0732,0.0216,0.0572,0.0685,0.1047,0.1869,0.2409
INT8,6.76,256.0,0.0690,0.0068,0.0614,0.0678,0.0751,0.1086,0.1261


In [10]:
# ============================================================
# Cell - Pi Benchmark Results: Hybrid Stack
# ============================================================

import json
import pandas as pd

with open("pi_results_stack.json", "r") as f:
    stack = json.load(f)

hs    = stack["hybrid_stack"]
lat   = hs["latency"]
sizes = hs["file_sizes_kb"]

print(f"Timestamp : {stack['timestamp']}")
print(f"Platform  : {stack['platform']}  |  CPU Cores: {stack['cpu_cores']}  |  RAM: {stack['total_ram_mb']:.2f} MB")

size_df = pd.DataFrame([{
    "Family 1 (KB)" : sizes["family1"],
    "Family 2 (KB)" : sizes["family2"],
    "Family 3 (KB)" : sizes["family3"],
    "Meta (KB)"     : sizes["meta"],
    "Total (KB)"    : sizes["total"],
}])

print("\n── Hybrid Stack File Sizes ───────────────────────────────────")
display(size_df)

stack_df = pd.DataFrame([{
    "Model"      : "Hybrid Stack",
    "Size (KB)"  : sizes["total"],
    "RAM (KB)"   : hs["ram_kb"],
    "Mean (ms)"  : lat["mean_ms"],
    "Std (ms)"   : lat["std_ms"],
    "Min (ms)"   : lat["min_ms"],
    "p50 (ms)"   : lat["p50_ms"],
    "p95 (ms)"   : lat["p95_ms"],
    "p99 (ms)"   : lat["p99_ms"],
    "Max (ms)"   : lat["max_ms"],
}]).set_index("Model")

print("\n── Hybrid Stack Benchmark ────────────────────────────────────")
display(stack_df)


Timestamp : 2026-03-03 07:36:12
Platform  : aarch64  |  CPU Cores: 4  |  RAM: 3796.81 MB

── Hybrid Stack File Sizes ───────────────────────────────────


,Family 1 (KB),Family 2 (KB),Family 3 (KB),Meta (KB),Total (KB)
0,1978.28,83.02,1168.64,0.56,3230.5



── Hybrid Stack Benchmark ────────────────────────────────────


,Size (KB),RAM (KB),Mean (ms),Std (ms),Min (ms),p50 (ms),p95 (ms),p99 (ms),Max (ms)
Model,,,,,,,,,
Hybrid Stack,3230.5,1624.0,11.9351,2.7995,10.8922,11.1352,19.5028,24.3485,29.3052


In [11]:
# ============================================================
# Cell - Combined Benchmark: ONNX Models + Hybrid Stack
# ============================================================

import json
import pandas as pd

with open("pi_results_onnx.json", "r") as f:
    onnx = json.load(f)
with open("pi_results_stack.json", "r") as f:
    stack = json.load(f)

def extract_row(label, size_kb, ram_kb, lat):
    return {
        "Model"      : label,
        "Size (KB)"  : size_kb,
        "RAM (KB)"   : ram_kb,
        "Mean (ms)"  : lat["mean_ms"],
        "Std (ms)"   : lat["std_ms"],
        "Min (ms)"   : lat["min_ms"],
        "p50 (ms)"   : lat["p50_ms"],
        "p95 (ms)"   : lat["p95_ms"],
        "p99 (ms)"   : lat["p99_ms"],
        "Max (ms)"   : lat["max_ms"],
    }

rows = [
    extract_row("sklearn (SKLearn)",  onnx["sklearn"]["size_kb"],              onnx["sklearn"]["ram_kb"],  onnx["sklearn"]["latency"]),
    extract_row("fp32 (ONNX)",        onnx["fp32"]["size_kb"],                 onnx["fp32"]["ram_kb"],     onnx["fp32"]["latency"]),
    extract_row("int8 (ONNX)",        onnx["int8"]["size_kb"],                 onnx["int8"]["ram_kb"],     onnx["int8"]["latency"]),
    extract_row("Hybrid Stack",       stack["hybrid_stack"]["file_sizes_kb"]["total"], stack["hybrid_stack"]["ram_kb"], stack["hybrid_stack"]["latency"]),
]

combined_df = pd.DataFrame(rows).set_index("Model")

print("=" * 70)
print("COMBINED BENCHMARK RESULTS — Raspberry Pi")
print("=" * 70)
# display(combined_df.style.highlight_min(axis=0, subset=["Mean (ms)", "Size (KB)", "RAM (KB)"], color="lightgreen")
#                          .highlight_max(axis=0, subset=["Mean (ms)", "Size (KB)", "RAM (KB)"], color="lightsalmon")
#                          .format("{:.4f}", subset=["Mean (ms)", "Std (ms)", "Min (ms)", "p50 (ms)", "p95 (ms)", "p99 (ms)", "Max (ms)"])
#                          .format("{:.2f}", subset=["Size (KB)", "RAM (KB)"]))

display(combined_df)

COMBINED BENCHMARK RESULTS — Raspberry Pi


,Size (KB),RAM (KB),Mean (ms),Std (ms),Min (ms),p50 (ms),p95 (ms),p99 (ms),Max (ms)
Model,,,,,,,,,
sklearn (SKLearn),90.06,378.0,5.9632,28.8927,1.3985,1.9428,4.2846,233.1406,294.3745
fp32 (ONNX),13.98,288.0,0.0732,0.0216,0.0572,0.0685,0.1047,0.1869,0.2409
int8 (ONNX),6.76,256.0,0.0690,0.0068,0.0614,0.0678,0.0751,0.1086,0.1261
Hybrid Stack,3230.50,1624.0,11.9351,2.7995,10.8922,11.1352,19.5028,24.3485,29.3052
